# Initialisation-sensitivity check: imports and config hash

In [1]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("KAGGLE_KEY")
secret_value_1 = user_secrets.get_secret("KAGGLE_USERNAME")

In [2]:
import sys
import os
import glob
import shutil
import warnings
import numpy as np
import pandas as pd
import torch

warnings.filterwarnings(
    "ignore",
    message="The behavior of DataFrame concatenation with empty or all-NA entries is deprecated",
    category=FutureWarning,
)

try:
    from kaggle_secrets import UserSecretsClient
    _sec = UserSecretsClient()
    os.environ["KAGGLE_USERNAME"] = _sec.get_secret("KAGGLE_USERNAME")
    os.environ["KAGGLE_KEY"] = _sec.get_secret("KAGGLE_KEY")
    print("kaggle credentials loaded from secrets")
except Exception as exc:
    print("kaggle secrets not loaded:", exc)

try:
    import core
except ModuleNotFoundError:
    hits = glob.glob("/kaggle/input/**/core.py", recursive=True)
    if not hits:
        raise ModuleNotFoundError("core.py not found.")
    sys.path.insert(0, os.path.dirname(hits[0]))
    import core

print("import-time CONFIG_HASH:", core.CONFIG_HASH)

kaggle credentials loaded from secrets
import-time CONFIG_HASH: 03fa342b535795c1


# GPU check

In [3]:
if torch.cuda.is_available():
    print("CUDA available:", torch.cuda.get_device_name(0))
else:
    print("WARNING: no CUDA device. This check is 20 small logistic/cnn runs, "
          "so it may proceed on CPU, but it will be slow. No ResNet-18 here.")

CUDA available: Tesla T4


# Stage the frozen config, refresh, then load data

In [4]:
pulled_ok = False
if core.DATAWISE_RESULTS_SLUG is not None:
    try:
        core.pull_results_store()
        pulled_ok = True
        print("pulled results store (latest) from", core.DATAWISE_RESULTS_SLUG)
    except Exception as exc:
        print("PULL FAILED:", exc)

if not pulled_ok:
    raise RuntimeError(
        "pull failed. This session would otherwise push a fresh store over the "
        "existing one and clobber prior protocols. Fix the pull and rerun. Do not "
        "let the grid or push proceed without the prior ledger in place."
    )

if not os.path.exists(core.PATHS["ledger"]):
    raise RuntimeError(
        f"pull reported success but no ledger at {core.PATHS['ledger']}. The store "
        "is empty or malformed. Stop and check the dataset before running the grid."
    )

prior = pd.read_csv(core.PATHS["ledger"])
protocols = sorted(set("|".join(prior["protocol"].astype(str)).split("|")))
print(f"prior ledger rows: {len(prior)}")
print("protocols present:", protocols)

required = {"protocolA", "protocolB", "protocolC", "protocolD"}
missing = required - set(protocols)
if missing:
    raise RuntimeError(
        f"pulled ledger is missing {missing}. This is not the full post-sweep state. "
        "Confirm 01-04 finished and pushed, then rerun."
    )
print("prior state confirmed: A, B, C, D present")

frozen_target = core.PATHS["frozen_config"]
if not os.path.exists(frozen_target):
    hits = glob.glob("/kaggle/input/**/frozen_config.md", recursive=True)
    if not hits:
        raise FileNotFoundError(
            "frozen_config.md not found in the working dir or any mounted dataset. "
            "Run 00_setup.ipynb and mount the results store (or the dataset holding "
            "frozen_config.md) before running this notebook."
        )
    core._ensure(os.path.dirname(frozen_target))
    shutil.copy(hits[0], frozen_target)
    print("staged frozen_config.md from", hits[0])

h = core.refresh_config()
print("active CONFIG_HASH after refresh:", h)
print("core.NORM_MEAN now:", core.NORM_MEAN)
print("core.NORM_STD  now:", core.NORM_STD)

data = core.load_pcam()

Dataset URL: https://www.kaggle.com/datasets/hrushisanap/datawise-results
pulled results store (latest) from hrushisanap/datawise-results
prior ledger rows: 225
protocols present: ['protocolA', 'protocolB', 'protocolC', 'protocolD']
prior state confirmed: A, B, C, D present
active CONFIG_HASH after refresh: 03fa342b535795c1
core.NORM_MEAN now: [0.700756, 0.538358, 0.691621]
core.NORM_STD  now: [0.234978, 0.277409, 0.212895]
[20:17:00] train pos/neg 131072/131072
[20:17:00] valid pos/neg (raw) 16369/16399
[20:17:00] test pos/neg (raw) 16377/16391
[20:17:01] valid balanced 32738 (16369/16369)
[20:17:01] test balanced 32754 (16377/16377)


# Build the init-sensitivity run grid

In [5]:
CONDITIONS = [(1000, 1000), (2000, 2000)]
INIT_SEEDS = [21, 22, 23, 24, 25]
MODELS = ["logistic", "cnn"]
FIXED_DATA_SEED = 21
FIXED_LOADER_SEED = 21

n_pos = len(data.train_pos_idx)
n_neg = len(data.train_neg_idx)

print(f"{'idx':>3}  {'model':>9}  {'n_min':>6}  {'n_maj':>6}  "
      f"{'data':>4}  {'init':>4}  {'loader':>6}")
idx = 0
for model in MODELS:
    for (n_min, n_maj) in CONDITIONS:
        assert n_min <= n_pos, f"n_min {n_min} exceeds positive pool {n_pos}"
        assert n_maj <= n_neg, f"n_maj {n_maj} exceeds negative pool {n_neg}"
        for init_seed in INIT_SEEDS:
            print(f"{idx:>3}  {model:>9}  {n_min:>6}  {n_maj:>6}  "
                  f"{FIXED_DATA_SEED:>4}  {init_seed:>4}  {FIXED_LOADER_SEED:>6}")
            idx += 1

print(f"\n{len(MODELS)} models x {len(CONDITIONS)} conditions x "
      f"{len(INIT_SEEDS)} init seeds -> "
      f"{len(MODELS) * len(CONDITIONS) * len(INIT_SEEDS)} run_training calls")

idx      model   n_min   n_maj  data  init  loader
  0   logistic    1000    1000    21    21      21
  1   logistic    1000    1000    21    22      21
  2   logistic    1000    1000    21    23      21
  3   logistic    1000    1000    21    24      21
  4   logistic    1000    1000    21    25      21
  5   logistic    2000    2000    21    21      21
  6   logistic    2000    2000    21    22      21
  7   logistic    2000    2000    21    23      21
  8   logistic    2000    2000    21    24      21
  9   logistic    2000    2000    21    25      21
 10        cnn    1000    1000    21    21      21
 11        cnn    1000    1000    21    22      21
 12        cnn    1000    1000    21    23      21
 13        cnn    1000    1000    21    24      21
 14        cnn    1000    1000    21    25      21
 15        cnn    2000    2000    21    21      21
 16        cnn    2000    2000    21    22      21
 17        cnn    2000    2000    21    23      21
 18        cnn    2000    2000 

# Run the grid (fixed data and loader seeds, vary init seed)

In [6]:
done = 0
for model in MODELS:
    for (n_min, n_maj) in CONDITIONS:
        for init_seed in INIT_SEEDS:
            run_id = core.run_training(
                "init_sensitivity", model, n_min, n_maj,
                FIXED_DATA_SEED, init_seed, FIXED_LOADER_SEED, data,
                target_ratio="", resume=True)
            done += 1
            print(f"[{done:>2}/20] {model} m={n_min} M={n_maj} "
                  f"init={init_seed} -> {run_id}")

print("grid calls completed:", done)

[20:17:01] [init_sensitivity][logistic][m=1000 M=1000 r=1.00][seed=21] reuse of logistic__m1000_M1000__d21_i21_l21, no retrain
[ 1/20] logistic m=1000 M=1000 init=21 -> logistic__m1000_M1000__d21_i21_l21
[20:17:01] logistic parameter count: 55298
[20:17:11] [init_sensitivity][logistic][m=1000 M=1000 r=1.00][seed=21] step 250/5000 loss=0.6458
[20:17:21] [init_sensitivity][logistic][m=1000 M=1000 r=1.00][seed=21] step 500/5000 loss=0.4941
[20:17:30] [init_sensitivity][logistic][m=1000 M=1000 r=1.00][seed=21] step 750/5000 loss=0.3551
[20:17:39] [init_sensitivity][logistic][m=1000 M=1000 r=1.00][seed=21] step 1000/5000 loss=0.3389
[20:17:49] [init_sensitivity][logistic][m=1000 M=1000 r=1.00][seed=21] step 1250/5000 loss=0.6661
[20:17:58] [init_sensitivity][logistic][m=1000 M=1000 r=1.00][seed=21] step 1500/5000 loss=0.2655
[20:18:07] [init_sensitivity][logistic][m=1000 M=1000 r=1.00][seed=21] step 1750/5000 loss=0.3910
[20:18:17] [init_sensitivity][logistic][m=1000 M=1000 r=1.00][seed=21]

# Session summary from the ledger (no test metrics)

In [7]:
if not os.path.exists(core.PATHS["ledger"]):
    raise FileNotFoundError("no ledger yet; run the grid cell first")

ledger = pd.read_csv(core.PATHS["ledger"])
mask = ledger["protocol"].astype(str).str.contains("init_sensitivity")
rows = ledger[mask]

order = {"failed": 0, "success": 1}
rows = rows.assign(_rank=rows["status"].map(order).fillna(-1))
dedup = rows.sort_values("_rank").drop_duplicates("run_id", keep="last").drop(columns="_rank")
n_success = int((dedup["status"] == "success").sum())
n_failed = int((dedup["status"] == "failed").sum())
expected = len(MODELS) * len(CONDITIONS) * len(INIT_SEEDS)

print(f"Init sensitivity: {n_success} success, {n_failed} failed, "
      f"{expected - n_success} pending of {expected}")

failed_rows = dedup[dedup["status"] == "failed"]
if len(failed_rows):
    print("\nfailed runs:")
    for _, r in failed_rows.iterrows():
        print(f"  {r['run_id']}  (model={r['model']} "
              f"n_min={r['n_min']} n_maj={r['n_maj']} init_seed={r['init_seed']})")

core._ensure(core.PATHS["summaries"])
summary_path = os.path.join(core.PATHS["summaries"], "06_init_sensitivity.md")
lines = [
    "# Notebook 06: initialisation-sensitivity check session summary",
    "",
    f"Active config hash: {core.CONFIG_HASH}",
    "",
    f"- expected unique runs: {expected} (2 models x 2 conditions x 5 init seeds)",
    f"- success: {n_success}",
    f"- failed: {n_failed}",
    f"- pending: {expected - n_success}",
    "- this check isolates initialisation noise: data_seed and loader_seed are fixed at 21 "
    "and only init_seed varies over 21-25. It does not measure data-sampling variance.",
    "- the init_seed=21 runs share run_ids with main-experiment runs and are reused via resume, "
    "not retrained.",
    "",
    "## Failed runs",
]
if len(failed_rows):
    for _, r in failed_rows.iterrows():
        lines.append(f"- {r['run_id']} (model={r['model']} "
                     f"n_min={r['n_min']} n_maj={r['n_maj']} init_seed={r['init_seed']})")
else:
    lines.append("- none")
lines.append("")
with open(summary_path, "w") as f:
    f.write("\n".join(lines))
print("\nwrote", summary_path)

Init sensitivity: 20 success, 0 failed, 0 pending of 20

wrote /kaggle/working/summaries/06_init_sensitivity.md


# Persistence 

In [8]:
if core.DATAWISE_RESULTS_SLUG is not None:
    core.push_results_store()
    print("pushed results store to", core.DATAWISE_RESULTS_SLUG)
else:
    print("DATAWISE_RESULTS_SLUG is unset; persistence skipped this session.")
    print("WARNING: without it, the ledger and logits will not survive session end.")

Starting upload for file subsets.zip


100%|██████████| 11.0M/11.0M [00:00<00:00, 13.9MB/s]


Upload successful: subsets.zip (11MB)
Starting upload for file logits.zip


100%|██████████| 74.1M/74.1M [00:01<00:00, 50.5MB/s]


Upload successful: logits.zip (74MB)
Starting upload for file isic.zip


...

Upload successful: results.zip (5KB)
Starting upload for file random_permutations.zip


100%|██████████| 2.25M/2.25M [00:00<00:00, 2.90MB/s]


Upload successful: random_permutations.zip (2MB)
Starting upload for file logs.zip


100%|██████████| 61.9k/61.9k [00:00<00:00, 90.8kB/s]


Upload successful: logs.zip (62KB)
pushed results store to hrushisanap/datawise-results
